# 🎬 Video2X trên Kaggle (AI Video Upscaling & Frame Interpolation)

Notebook này được tối ưu để chạy **[Video2X v6.2.0](https://github.com/k4yt3x/video2x)** (phiên bản C++ native hiệu năng cao) trên nền tảng **Kaggle** sử dụng GPU miễn phí (**NVIDIA Tesla T4** hoặc **P100**).

---
### ⚠️ Các cài đặt bắt buộc trên Kaggle trước khi chạy:
1. **Bật GPU:** Ở bảng **Notebook Settings** bên phải màn hình ➡️ mục **Accelerator** ➡️ chọn **GPU T4 x2** (hoặc GPU P100).
2. **Bật Internet:** Ở bảng **Notebook Settings** ➡️ gạt toggle **Internet** sang **ON** (để tải package Video2X và mô hình AI).
3. **Quản lý file:**
   - Video sau khi xử lý sẽ lưu tại `/kaggle/working/output/`.
   - Bạn có thể tải video về máy trực tiếp từ tab **Output** ở panel bên phải hoặc bấm vào link tải ngay trong notebook.


## 🖥️ Bước 0: Kiểm tra phần cứng GPU
Chạy lệnh sau để xác nhận Kaggle đã cấp phát GPU NVIDIA và driver đang hoạt động.


In [ ]:
# Kiểm tra thông tin GPU được cấp phát
!nvidia-smi


## 📦 Bước 1: Cài đặt Video2X v6.2.0 & Cấu hình Vulkan cho Kaggle
Video2X v6 sử dụng **Vulkan API** và **ncnn** để xử lý video trực tiếp qua GPU mà không tốn dung lượng ổ cứng tạm.
Trên Kaggle, chúng ta cần cài đặt `libvulkan`, `ffmpeg` và tạo file cấu hình `nvidia_icd.json` để Vulkan nhận diện đúng GPU NVIDIA.


In [ ]:
import os
import json
import subprocess
from pathlib import Path

print("⏳ [1/4] Đang cài đặt các thư viện hệ thống (Vulkan, FFmpeg)...")
!apt-get update -yqq
!apt-get install -yqq --no-install-recommends libvulkan1 libvulkan-dev vulkan-tools ffmpeg

print("⏳ [2/4] Đang tải và cài đặt Video2X v6.2.0 (Debian package)...\n")
deb_file = "/tmp/video2x-linux-ubuntu2204-amd64.deb"
!wget -q --show-progress -O {deb_file} https://github.com/k4yt3x/video2x/releases/download/6.2.0/video2x-linux-ubuntu2204-amd64.deb
!dpkg -i {deb_file} || apt-get install -fyqq

print("⏳ [3/4] Cấu hình NVIDIA Vulkan ICD cho môi trường Kaggle...")
icd_dir = Path("/usr/share/vulkan/icd.d")
icd_dir.mkdir(parents=True, exist_ok=True)
Path("/etc/vulkan/icd.d").mkdir(parents=True, exist_ok=True)

nvidia_icd = {
    "file_format_version": "1.0.0",
    "ICD": {
        "library_path": "libGLX_nvidia.so.0",
        "api_version": "1.3.0"
    }
}

with open(icd_dir / "nvidia_icd.json", "w") as f:
    json.dump(nvidia_icd, f, indent=4)

!cp /usr/share/vulkan/icd.d/nvidia_icd.json /etc/vulkan/icd.d/nvidia_icd.json

# Thiết lập biến môi trường chỉ định Vulkan sử dụng driver NVIDIA
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"

print("⏳ [4/4] Kiểm tra Vulkan Device & Video2X version...")
!vulkaninfo --summary 2>/dev/null | grep -E "deviceName|driverName" || !vulkaninfo | grep -i "device name"
!video2x --version

print("\n✅ Cài đặt hoàn tất! Sẵn sàng xử lý video.")


## 📥 Bước 2: Chuẩn bị Video đầu vào
Bạn có 3 cách để nạp video vào Kaggle:
* **Cách 1 (Khuyên dùng):** Thêm Dataset chứa video qua nút **+ Add Input** (góc trên bên phải) ➡️ Video nằm tại `/kaggle/input/`.
* **Cách 2:** Tải trực tiếp video từ direct URL bằng ô nhập link bên dưới.
* **Cách 3:** Đặt video vào thư mục `/kaggle/working/input/`.

Chạy cell bên dưới để quét các video sẵn có hoặc nhập link tải trực tiếp.


In [ ]:
import os
import pathlib
import ipywidgets as widgets
from IPython.display import display

INPUT_ROOT = pathlib.Path("/kaggle/input")
WORKING_INPUT = pathlib.Path("/kaggle/working/input")
WORKING_INPUT.mkdir(parents=True, exist_ok=True)

selected_input_file = None
video_extensions = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv", ".ts"}

# Quét các video có sẵn
found_videos = []
for p in list(INPUT_ROOT.rglob("*")) + list(WORKING_INPUT.glob("*")):
    if p.is_file() and p.suffix.lower() in video_extensions:
        found_videos.append(p)

print(f"🎬 Tìm thấy {len(found_videos)} file video trong môi trường.")

# Dropdown chọn video
dropdown_opts = [str(f) for f in found_videos] if found_videos else ["(Chưa tìm thấy video)"]
dropdown = widgets.Dropdown(
    options=dropdown_opts,
    description="Chọn video:",
    layout=widgets.Layout(width="80%")
)

def on_dropdown_change(change):
    global selected_input_file
    if change["new"] and change["new"] != "(Chưa tìm thấy video)":
        selected_input_file = pathlib.Path(change["new"])
        print(f"🎯 Đã chọn: {selected_input_file}")

dropdown.observe(on_dropdown_change, names="value")
if found_videos:
    selected_input_file = found_videos[0]
    print(f"🎯 Mặc định chọn: {selected_input_file}")

display(dropdown)

# Hộp tải video từ link trực tiếp
url_input = widgets.Text(
    placeholder="Dán direct link video (.mp4, .mkv, ...) vào đây...",
    description="Tải từ URL:",
    layout=widgets.Layout(width="65%")
)
download_btn = widgets.Button(description="Tải Video", button_style="success")

def on_download_click(b):
    global selected_input_file
    url = url_input.value.strip()
    if not url:
        print("⚠️ Vui lòng nhập link URL trước khi bấm tải!")
        return
    raw_name = url.split("?")[0].split("/")[-1]
    filename = raw_name if any(raw_name.endswith(e) for e in video_extensions) else "video_input.mp4"
    target = WORKING_INPUT / filename
    print(f"⏳ Đang tải: {url} -> {target}...")
    res = os.system(f"wget -q --show-progress -O '{target}' '{url}'")
    if res == 0 and target.exists():
        selected_input_file = target
        print(f"✅ Tải hoàn tất! Đã chọn video: {selected_input_file}")
    else:
        print("❌ Lỗi khi tải video từ URL. Hãy kiểm tra lại link.")

download_btn.on_click(on_download_click)
display(widgets.HBox([url_input, download_btn]))


## ⚙️ Bước 3: Cấu hình và Thực thi Video Upscale

### 💡 Hướng dẫn chọn tham số:
* **`PROCESSOR`**:
  * `realesrgan`: Nâng độ phân giải bằng Deep Learning với chất lượng vượt trội.
  * `libplacebo`: Dùng bộ shader Anime4K GLSL, tốc độ cực kỳ nhanh cho anime/hoạt hình.
* **`REALESRGAN_MODEL`**:
  * `realesr-animevideov3`: Chuyên dụng cho hoạt hình/anime (hỗ trợ scale 2x, 3x, 4x) - **Khuyên dùng**.
  * `realesrgan-plus-anime`: Chất lượng cực cao cho anime (chỉ hỗ trợ scale 4x).
  * `realesrgan-plus`: Tối ưu cho video đời thực, người thật, phong cảnh (chỉ hỗ trợ 4x).
* **`CRF` (Chất lượng video xuất)**: Giá trị từ 18 (rất nét, file to) đến 24 (dung lượng nhẹ hơn). Mặc định `20` là mức tối ưu.


In [ ]:
import os
import time
import pathlib
import subprocess

# ==========================================================
# ⚙️ TÙY CHỈNH THÔNG SỐ UPSCALING TẠI ĐÂY
# ==========================================================
PROCESSOR = "realesrgan"            # "realesrgan" hoặc "libplacebo"

# Cấu hình Real-ESRGAN
REALESRGAN_MODEL = "realesr-animevideov3"  # "realesr-animevideov3", "realesrgan-plus-anime", "realesrgan-plus"
SCALE = 2                                  # 2, 3, hoặc 4 (Model animevideov3 hỗ trợ 2, 3, 4; model khác chỉ 4)

# Cấu hình libplacebo (Anime4K)
SHADER = "anime4k-v4-a"             # "anime4k-v4-a", "anime4k-v4-b", "anime4k-v4-c", "anime4k-v4.1-gan"
TARGET_WIDTH = 1920                 # Đặt 0 nếu muốn tự tính toán
TARGET_HEIGHT = 1080                # Đặt 0 nếu muốn tự tính toán

# Cấu hình FFmpeg Encoder
CODEC = "libx264"                   # "libx264" (CPU x264 chất lượng cao) hoặc "h264_nvenc" (NVIDIA Hardware)
PRESET = "slow"                    # "ultrafast", "fast", "medium", "slow", "veryslow"
CRF = 20                            # 17 - 23 (số càng thấp chất lượng càng cao)
PIXFMT = "auto"                     # "auto", "yuv420p"
BITRATE = 0                         # 0 để dùng CRF
LOGLEVEL = "info"                   # "info", "debug", "warn", "error"

# Thư mục lưu kết quả
OUTPUT_DIR = pathlib.Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ==========================================================

# Kiểm tra đầu vào
if 'selected_input_file' not in globals() or selected_input_file is None or not pathlib.Path(selected_input_file).exists():
    raise ValueError("❌ Bạn chưa chọn video đầu vào! Vui lòng hoàn thành Bước 2 trước.")

input_path = pathlib.Path(selected_input_file)
output_path = OUTPUT_DIR / f"{input_path.stem}_upscaled{input_path.suffix}"

# Xóa file cũ nếu đã tồn tại để tránh xung đột
if output_path.exists():
    output_path.unlink()

print(f"🎬 Video nguồn: {input_path}")
print(f"💾 Video đích  : {output_path}")
print(f"🚀 Processor  : {PROCESSOR} | Scale: {SCALE}x")

# Xây dựng lệnh video2x
cmd = [
    "video2x",
    "--input", f'"{input_path}"',
    "--output", f'"{output_path}"',
    "--processor", PROCESSOR,
    "--codec", CODEC,
    "--log-level", LOGLEVEL,
    "-e", f"preset={PRESET}",
    "-e", f"crf={CRF}"
]

if BITRATE > 0:
    cmd.extend(["--bit-rate", str(BITRATE)])

if PIXFMT != "auto":
    cmd.extend(["--pix-fmt", PIXFMT])

if PROCESSOR == "realesrgan":
    cmd.extend([
        "--realesrgan-model", REALESRGAN_MODEL,
        "--scaling-factor", str(SCALE)
    ])
elif PROCESSOR == "libplacebo":
    cmd.extend(["--libplacebo-shader", f'"{SHADER}"'])
    if TARGET_WIDTH > 0:
        cmd.extend(["--width", str(TARGET_WIDTH)])
    if TARGET_HEIGHT > 0:
        cmd.extend(["--height", str(TARGET_HEIGHT)])

full_cmd = " ".join(cmd)
print("-" * 60)
print(f"Executing: {full_cmd}")
print("-" * 60)

start_time = time.time()
p = subprocess.Popen(full_cmd, shell=True)
p.communicate()
elapsed = time.time() - start_time

if p.returncode == 0 and output_path.exists():
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print("=" * 60)
    print(f"🎉 XỬ LÝ THÀNH CÔNG trong {elapsed:.1f} giây!")
    print(f"📁 File kết quả: {output_path} ({size_mb:.2f} MB)")
    print("=" * 60)
else:
    print("=" * 60)
    print(f"❌ Đã xảy ra lỗi khi xử lý! (Exit code: {p.returncode})")
    print("=" * 60)


## 📺 Bước 4: Xem trước (Preview) & Tải Video
Sau khi xử lý xong, bạn có thể xem trước video trực tiếp ngay trên notebook và nhấp vào liên kết tải xuống.


In [ ]:
from IPython.display import display, HTML, FileLink
import pathlib

if 'output_path' in globals() and output_path.exists():
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"✅ Video sẵn sàng tải về: {output_path.name} ({size_mb:.2f} MB)\n")
    
    # 1. Hiển thị link tải trực tiếp
    print("👇 Bấm vào liên kết bên dưới để tải file về máy:")
    display(FileLink(str(output_path)))
    
    # 2. Preview video player nếu là file MP4
    if output_path.suffix.lower() == ".mp4":
        display(HTML(f"""
        <div style="margin-top: 15px;">
            <h4>🎬 Xem trước video:</h4>
            <video width="640" height="360" controls autoplay muted>
                <source src="{output_path}" type="video/mp4">
                Trình duyệt của bạn không hỗ trợ thẻ video HTML5.
            </video>
        </div>
        """))
else:
    print("⚠️ Chưa tìm thấy file video kết quả. Hãy chạy Bước 3 trước.")


## ⏩ Bước 5 (Tùy chọn): Tăng FPS với mô hình RIFE
Video2X hỗ trợ mô hình **RIFE (Real-Time Intermediate Flow Estimation)** để nhân số khung hình (ví dụ nhân đôi từ 24fps/30fps lên 60fps), giúp chuyển động mượt mà hơn đáng kể.


In [ ]:
import os
import time
import pathlib
import subprocess

# Cấu hình RIFE
RIFE_MODEL = "rife-v4.6"       # "rife-v4.6", "rife-anime", "rife-v4", "rife-UHD", "rife-v2.4"
FRAME_RATE_MUL = 2             # Hệ số nhân FPS (2: nhân đôi, 4: nhân 4)
CODEC = "libx264"
PRESET = "slow"
CRF = 20

OUTPUT_DIR = pathlib.Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if 'selected_input_file' not in globals() or not pathlib.Path(selected_input_file).exists():
    raise ValueError("❌ Chưa chọn video đầu vào!")

input_path = pathlib.Path(selected_input_file)
output_rife = OUTPUT_DIR / f"{input_path.stem}_rife_{FRAME_RATE_MUL}x{input_path.suffix}"

if output_rife.exists():
    output_rife.unlink()

rife_cmd = [
    "video2x",
    "--input", f'"{input_path}"',
    "--output", f'"{output_rife}"',
    "--processor", "rife",
    "--rife-model", RIFE_MODEL,
    "--frame-rate-mul", str(FRAME_RATE_MUL),
    "--codec", CODEC,
    "--log-level", "info",
    "-e", f"preset={PRESET}",
    "-e", f"crf={CRF}"
]

full_rife_cmd = " ".join(rife_cmd)
print(f"🚀 Đang chạy RIFE frame interpolation:\n{full_rife_cmd}\n" + "-"*60)

start_t = time.time()
res = os.system(full_rife_cmd)
dur = time.time() - start_t

if res == 0 and output_rife.exists():
    mb = output_rife.stat().st_size / (1024 * 1024)
    print("=" * 60)
    print(f"🎉 Hoàn tất RIFE trong {dur:.1f} giây!")
    print(f"📁 File kết quả: {output_rife} ({mb:.2f} MB)")
    print("=" * 60)
else:
    print("=" * 60)
    print("⚠️ RIFE không hoàn thành thành công (kiểm tra lại log chi tiết ở trên).")
    print("=" * 60)
